# Fifteen drinks: a map from customer-group ratings

**Multidimensional Scaling** · a worked example from the StatNex teaching guide

**The story.** A convenience-store chain surveyed how much ten customer groups (men and women in five age bands) like 15 drinks, on a 1-9 scale. It wants a map showing which drinks appeal to the same customers.

**The question.** Which drinks appeal to the same customer groups, and what are the main lines that divide them?

**To run it:**

1. Choose **Copy to Drive** at the top (or **File → Save a copy in Drive**). Your copy opens in a new tab. It is yours, so Colab runs it without a warning, and it keeps your changes.
2. In your copy, choose **Runtime → Run all**.

You can also run this page without a copy. Colab then warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

This analysis needs `scikit-learn`. This cell installs them. If Colab asks you to restart the session, restart it and choose **Run all** again.

In [ ]:
%pip install -q "scikit-learn>=1.8"

## The table

This cell saves the table `09_multivariate/multidimensional_scaling.csv` (15 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Drink,Men 18-24,Men 25-34,Men 35-44,Men 45-54,Men 55+,Women 18-24,Women 25-34,Women 35-44,Women 45-54,Women 55+
Cola,6.71,6.67,5.06,4.8,3.89,6.74,6.2,5.75,4.41,4.13
Diet cola,5.59,6.05,5.46,6.15,5.2,6.38,5.9,5.81,5.89,5.54
Orange juice,5.93,6.05,5.67,5.6,5.46,6.75,6.63,6.45,5.9,5.74
Green tea,4.98,5.41,6.36,7.34,8.09,5.47,6.14,6.65,8.23,8.48
Black coffee,5.03,5.71,5.89,6.93,7.09,4.78,6.29,6.47,7.22,6.84
Latte,5.91,5.81,6.51,6.41,5.36,5.87,6.39,6.21,6.62,5.59
Energy drink,6.56,6.98,6.54,6.07,5.04,6.5,6.98,6.36,5.82,5.03
Sports drink,5.48,4.98,5.77,5.08,5.09,6.27,6.24,6.32,5.62,5.57
Bubble tea,6.76,6.57,5.17,4.76,4.28,6.71,6.07,5.24,4.2,3.36
Smoothie,5.95,5.91,5.57,6.19,6.38,6.55,6.49,6.83,6.95,6.27
Beer,5.59,6.2,7.06,6.6,6.33,3.89,5.05,5.58,5.75,5.6
Highball,5.67,6.32,7.44,6.91,6.28,4.84,5.32,5.66,5.92,5.12
Red wine,6.23,6.92,6.37,7.14,6.86,4.96,5.35,6.4,6.25,5.89
Mineral water,4.64,5.31,4.92,6.35,7.31,4.92,5.46,6.21,6.63,7.54
Hot chocolate,6.24,5.83,4.85,4.42,3.42,6.46,5.98,5.28,4.63,3.08

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform
from sklearn.manifold import ClassicalMDS

pd.set_option("display.width", 120)

## The data

In [ ]:
# Text and blank cells count as missing, and a case missing any variable is left out.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
variables = ["Men 18-24", "Men 25-34", "Men 35-44", "Men 45-54", "Men 55+", "Women 18-24", "Women 25-34", "Women 35-44", "Women 45-54", "Women 55+"]
label_column = ["Drink"]
numbers = data[variables].apply(pd.to_numeric, errors="coerce")
complete = numbers.notna().all(axis=1)
x = numbers[complete].to_numpy(float)

# The app lays this table out like its regression methods. There is no response
# variable here, so its two rows are always 0.
summary = pd.DataFrame(
    {"n": [len(x), 0, len(data) - len(x), 0, len(data)]},
    index=["Valid cases", "Response only missing", "Explanatory only missing",
           "Both missing", "Total"],
)
summary["share"] = summary["n"] / len(data)
print(summary)

## Descriptive statistics and correlations

In [ ]:
# Variance and standard deviation divide by n - 1.
describe = pd.DataFrame(
    {"n": len(x), "mean": x.mean(axis=0), "variance": x.var(axis=0, ddof=1),
     "SD": x.std(axis=0, ddof=1), "min": x.min(axis=0), "max": x.max(axis=0)},
    index=variables,
)
correlation = np.corrcoef(x, rowvar=False)
print()
print(describe)
print()
print(pd.DataFrame(correlation, index=variables, columns=variables).round(4))

## The objects to place

In [ ]:
# The cases are placed, each described by its values on the variables.
objects = x
labels = (data.loc[complete, label_column[0]].astype(str).tolist() if label_column
          else [str(i + 1) for i in range(len(x))])
n = len(objects)
dimensions = 2
print(f"\n{n} objects, {dimensions} dimension(s)")

## Distance matrix

In [ ]:
# How far apart every pair of objects is: SciPy's pdist() measures each pair once and
# squareform() lays the result out as a table. 0 on the diagonal; the bigger the
# number, the less alike the two objects are.
# The Euclidean distance: the straight-line distance between two objects' profiles.
distances = squareform(pdist(objects, "euclidean"))
print()
print(pd.DataFrame(distances, index=labels, columns=labels).round(4))

## Double-centred matrix (Torgerson)

In [ ]:
# B = -1/2 J D^2 J, with J the centring matrix. If the distances come from points,
# B holds the inner products of the centred points, so its eigenvectors are the map.
# scikit-learn's ClassicalMDS does this and the next step inside one call (below);
# they are written out here to print B and every eigenvalue, as the app does.
centring = np.eye(n) - np.ones((n, n)) / n
b = -0.5 * centring @ (distances**2) @ centring
print()
print(pd.DataFrame(b, index=labels, columns=labels).round(4))

## Eigenvalues and Mardia's criteria

In [ ]:
# One eigenvalue per dimension, largest first. They add up to the total spread of the
# objects (the squared distances summed over every pair, divided by n), so each one
# is the part of that spread its dimension carries. Read Mardia's criteria down the column: the
# value at dimension 2 is the share a two-dimensional map keeps; near 1, the map
# loses little.
#
# Mardia's criteria (Mardia 1978) divide the running total of the positive
# eigenvalues by the size of *all* of them: the first by the sum of their absolute
# values, the second by the sum of their squares. A negative eigenvalue is part of
# the distances no map can show, so it counts in the total. These are the first
# number of R's cmdscale() GOF and Stata's -mds- "Mardia fit measure 1 and 2".
# Against an eigenvalue that is not positive there is no dimension to keep, and the
# app prints "-" (NaN here).
eigenvalues, vectors = np.linalg.eigh((b + b.T) / 2)
order = np.argsort(-eigenvalues)
eigenvalues, vectors = eigenvalues[order], vectors[:, order]
# An eigenvalue no larger in size than 1e-10 times the largest is zero up to
# rounding: the solver returns zeros as noise of either sign, around 1e-15 of the
# largest. It is set to exactly 0, as the app does.
eigenvalues[np.abs(eigenvalues) <= 1e-10 * np.abs(eigenvalues).max()] = 0.0
kept = np.clip(eigenvalues, 0.0, None)
mardia = pd.DataFrame(
    {"eigenvalue": eigenvalues, "Mardia 1": kept.cumsum() / np.abs(eigenvalues).sum(),
     "Mardia 2": (kept**2).cumsum() / (eigenvalues**2).sum()},
    index=[f"Dim {j + 1}" for j in range(n)],
)
mardia.loc[eigenvalues <= 0, ["Mardia 1", "Mardia 2"]] = np.nan
print()
print(mardia)

# An axis and its mirror image are the same map, so each eigenvector is turned to make
# its first element positive. That choice also decides whether the map comes out
# mirrored against a real one: flip one axis (multiply it by -1) to undo it.
vectors = vectors * np.where(vectors[0] < 0, -1.0, 1.0)
print()
print(pd.DataFrame(vectors, index=labels, columns=mardia.index).round(4))

## Coordinates

In [ ]:
# Each kept eigenvector times the square root of its eigenvalue: scikit-learn's
# ClassicalMDS, given the distance table ("precomputed"), returns exactly these. It
# picks each axis's sign by its own rule, so the axes are turned to start positive
# as above. Two objects close on the map have similar profiles; the axes have no
# units of their own and are named by the objects at their two ends.
classical = ClassicalMDS(n_components=dimensions, metric="precomputed").fit(distances)
coordinates = classical.embedding_
coordinates = coordinates * np.where(coordinates[0] < 0, -1.0, 1.0)
dims = [f"Dim {j + 1}" for j in range(dimensions)]
print()
print(pd.DataFrame(coordinates, index=labels, columns=dims).round(4))

## Rotated map

In [ ]:
# Turned clockwise by the angle on the first two dimensions:
# x' = x cos(a) + y sin(a), y' = -x sin(a) + y cos(a). Distances do not change.
angle = 0
rotated = coordinates.copy()
if dimensions >= 2:
    a = np.deg2rad(angle)
    rotated[:, :2] = coordinates[:, :2] @ np.array([[np.cos(a), -np.sin(a)],
                                                    [np.sin(a), np.cos(a)]])
print(f"\nRotation angle: {angle} degrees")
print(pd.DataFrame(rotated, index=labels, columns=dims).round(4))

## What the result says

Multidimensional scaling (MDS) draws a map from distances. Here the distance between two drinks is the Euclidean (straight-line) distance between their ratings, so drinks that the ten customer groups rate alike sit close together.

The eigenvalues show how much each dimension of the map carries. By Mardia's first criterion, dimension 1 holds 0.72 of the total and the first two hold 0.87, so a flat map loses little. On dimension 1, hot chocolate (4.17), bubble tea (3.82) and cola (3.47) sit at one end and green tea (-4.60) at the other: the strong line, of age. On dimension 2, beer (-2.30) and highball (-2.09) stand apart: the weaker line, of gender. The ratings support these names: hot chocolate, bubble tea and cola get their highest ratings from the youngest groups and green tea from the oldest, and in every age band men rate beer and highball higher than women do. Cola and bubble tea are the closest pair, at a distance of 1.05.

The map shows who likes each drink, not why, and not whether neighbours compete.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.